# Análisis de errores

Usa `results/evaluation/test_predictions.csv` generado en
`final_evaluation.ipynb` — no necesita recargar ningún modelo.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('results/evaluation/test_predictions.csv')
df.head()

In [ ]:
label_col = 'is_humor'  # ajustar si la columna se llama distinto en el CSV
text_col  = 'text'

df['error_nbsvm']    = df[label_col] != df['pred_nbsvm']
df['error_baseline'] = df[label_col] != df['pred_baseline']
df['error_lora']     = df[label_col] != df['pred_lora']

## 1. Tasa de error por clase
% de falsos negativos sobre el total de humor real, % de falsos positivos
sobre el total de no-humor real — para cada uno de los 3 modelos.

In [ ]:
def error_rate_by_class(df, pred_col, label_col):
    humor_real = df[df[label_col] == 1]
    no_humor_real = df[df[label_col] == 0]

    fn_rate = (humor_real[pred_col] == 0).mean()       # humor real predicho como no-humor
    fp_rate = (no_humor_real[pred_col] == 1).mean()    # no-humor real predicho como humor
    return fn_rate, fp_rate

rows = []
for modelo, pred_col in [('NBSVM', 'pred_nbsvm'), ('BETO baseline', 'pred_baseline'), ('BETO + LoRA', 'pred_lora')]:
    fn_rate, fp_rate = error_rate_by_class(df, pred_col, label_col)
    rows.append({'modelo': modelo, 'tasa_falsos_negativos': fn_rate, 'tasa_falsos_positivos': fp_rate})

df_error_rates = pd.DataFrame(rows)

os.makedirs('results/evaluation', exist_ok=True)
df_error_rates.to_csv('results/evaluation/error_rates_by_class.csv', index=False)
df_error_rates

## 2. Falsos positivos y negativos más confiados (3 modelos)
Los que el modelo predijo con mayor seguridad pero estaban mal.

In [ ]:
def top_confident_errors(df, pred_col, prob_col, label_col, text_col, n=5):
    errores = df[df[label_col] != df[pred_col]].copy()

    fp = errores[errores[pred_col] == 1].nlargest(n, prob_col)
    fn = errores[errores[pred_col] == 0].nsmallest(n, prob_col)

    return fp[[text_col, label_col, pred_col, prob_col]], fn[[text_col, label_col, pred_col, prob_col]]

In [ ]:
print('=== NBSVM ===')
fp_nbsvm, fn_nbsvm = top_confident_errors(df, 'pred_nbsvm', 'prob_nbsvm', label_col, text_col)
print('\nFalsos positivos más confiados:')
display(fp_nbsvm)
print('\nFalsos negativos más confiados:')
display(fn_nbsvm)

In [ ]:
print('=== BETO baseline ===')
fp_baseline, fn_baseline = top_confident_errors(df, 'pred_baseline', 'prob_baseline', label_col, text_col)
print('\nFalsos positivos más confiados:')
display(fp_baseline)
print('\nFalsos negativos más confiados:')
display(fn_baseline)

In [ ]:
print('=== BETO + LoRA ===')
fp_lora, fn_lora = top_confident_errors(df, 'pred_lora', 'prob_lora', label_col, text_col)
print('\nFalsos positivos más confiados:')
display(fp_lora)
print('\nFalsos negativos más confiados:')
display(fn_lora)

## 3. Errores comunes a los 3 modelos
Tweets donde NBSVM, baseline y LoRA fallan los tres simultáneamente —
posibles "casos imposibles" (sarcasmo extremo, ambigüedad genuina).

In [ ]:
errores_comunes = df[df['error_nbsvm'] & df['error_baseline'] & df['error_lora']]
print(f'Tweets donde fallan los 3 modelos: {len(errores_comunes)} de {len(df)} ({100*len(errores_comunes)/len(df):.1f}%)')

errores_comunes[[text_col, label_col, 'pred_nbsvm', 'pred_baseline', 'pred_lora']].to_csv(
    'results/evaluation/errores_comunes_3_modelos.csv', index=False
)
errores_comunes[[text_col, label_col, 'pred_nbsvm', 'pred_baseline', 'pred_lora']]

## 4. Comparación cruzada NBSVM vs BETO vs LoRA
Dónde un modelo acierta y el otro falla.

In [ ]:
def comparacion_cruzada(df, error_col_a, error_col_b, nombre_a, nombre_b):
    solo_a_falla = df[df[error_col_a] & ~df[error_col_b]]
    solo_b_falla = df[~df[error_col_a] & df[error_col_b]]
    print(f'{nombre_a} falla y {nombre_b} acierta: {len(solo_a_falla)}')
    print(f'{nombre_b} falla y {nombre_a} acierta: {len(solo_b_falla)}')
    return solo_a_falla, solo_b_falla

In [ ]:
print('=== NBSVM vs BETO + LoRA ===')
nbsvm_falla, lora_falla = comparacion_cruzada(df, 'error_nbsvm', 'error_lora', 'NBSVM', 'LoRA')

print('\nEjemplos donde NBSVM falla y LoRA acierta (LoRA capta algo que NBSVM no):')
display(nbsvm_falla[[text_col, label_col, 'pred_nbsvm', 'pred_lora']].head(10))

print('\nEjemplos donde LoRA falla y NBSVM acierta:')
display(lora_falla[[text_col, label_col, 'pred_nbsvm', 'pred_lora']].head(10))

In [ ]:
print('=== BETO baseline vs BETO + LoRA ===')
baseline_falla, lora_falla_2 = comparacion_cruzada(df, 'error_baseline', 'error_lora', 'BETO baseline', 'LoRA')

print('\nEjemplos donde baseline falla y LoRA acierta (el fine-tuning eficiente ayuda):')
display(baseline_falla[[text_col, label_col, 'pred_baseline', 'pred_lora']].head(10))

## 5. Análisis por longitud de tweet
¿Los errores se concentran en tweets muy cortos o muy largos?
(Si no se ve nada interesante, esta sección se puede sacar del informe final.)

In [ ]:
df['longitud'] = df[text_col].str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (modelo, error_col) in zip(axes, [('NBSVM', 'error_nbsvm'), ('BETO baseline', 'error_baseline'), ('BETO + LoRA', 'error_lora')]):
    ax.hist(df[~df[error_col]]['longitud'], bins=30, alpha=0.6, label='Correcto', density=True)
    ax.hist(df[df[error_col]]['longitud'], bins=30, alpha=0.6, label='Error', density=True)
    ax.set_title(modelo)
    ax.set_xlabel('Longitud (caracteres)')
    ax.legend()
plt.tight_layout()
plt.savefig('results/evaluation/errores_por_longitud.png', dpi=150)
plt.show()

In [ ]:
for modelo, error_col in [('NBSVM', 'error_nbsvm'), ('BETO baseline', 'error_baseline'), ('BETO + LoRA', 'error_lora')]:
    print(f"{modelo}: longitud media correctos={df[~df[error_col]]['longitud'].mean():.1f} | "
          f"longitud media errores={df[df[error_col]]['longitud'].mean():.1f}")

## 6. Distribución de confianza completa
Histograma de probabilidades separado por correcto/incorrecto, no solo los extremos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (modelo, prob_col, error_col) in zip(
    axes,
    [('BETO baseline', 'prob_baseline', 'error_baseline'), ('BETO + LoRA', 'prob_lora', 'error_lora')]
):
    ax.hist(df[~df[error_col]][prob_col], bins=30, alpha=0.6, label='Correcto', density=True)
    ax.hist(df[df[error_col]][prob_col], bins=30, alpha=0.6, label='Error', density=True)
    ax.set_title(modelo)
    ax.set_xlabel('Probabilidad de humor')
    ax.legend()
plt.tight_layout()
plt.savefig('results/evaluation/distribucion_confianza.png', dpi=150)
plt.show()

## 7. Observaciones cualitativas

_Completar después de revisar los resultados arriba: patrones en los
errores comunes, qué tipo de humor capta LoRA que NBSVM no, si la
longitud importa, si los modelos están bien calibrados (errores cerca
de 0.5 vs. errores muy confiados)._